# Task 4 — Style-Conditioned Face-to-Sketch Generation (Colab runner)

A conditional GAN turns a face photograph plus a style choice (Style 1/2/3) into a pencil sketch: a pix2pix U-Net generator whose decoder is modulated by a learned style embedding, trained against a 70×70 PatchGAN discriminator that also sees the photo and the style. Run the cells **from top to bottom** on a **T4 GPU** (Runtime → Change runtime type → T4 GPU).

**If Colab disconnects:** reconnect, run cells 1 and 2 again, then re-run the cell that was interrupted. Everything is saved to Google Drive and resumes where it stopped: the Optuna study continues with the remaining trials, training continues from the last finished epoch (`--resume`), and a finished training cell returns immediately.

**Runtime (ESTIMATES, not measured on a T4):** about 1.5–2.5 h in total — Optuna ≈ 45–80 min (hard-capped at 80 min per session), final training ≈ 25–60 min, evaluation ≈ 2 min, ONNX export ≈ 1 min. The estimates come from CPU measurements (2.52 s/step at `base_channels=64`, batch 8, 112 steps per epoch) scaled by a typical T4 speed-up; **the real per-epoch time is printed in every training log line** — if an epoch takes much more than ~25 s, lower `--epochs` (e.g. 80) and re-run. Design decisions and all commands: `docs/fs2k_notes.md`.

## 1. Get the code

In [ ]:
# 1. code
REPO_URL = 'https://github.com/fsdev87/generative-ai-A1.git'
import os
if os.path.exists('/content/repo'):
    !git -C /content/repo pull
else:
    !git clone {REPO_URL} /content/repo
%cd /content/repo

## 2. Setup: Drive, packages, data cache, W&B, environment variables
Needs the Colab secret `WANDB_API_KEY`. `setup(fs2k=True)` also downloads FS2K and runs `scripts/prepare_fs2k.py`, which verifies the 2,104 photo/sketch pairs, checks the committed split (`manifests/fs2k_split.json`) and builds the 128×128 cache. Estimated time: 2–6 min in the first session (the FS2K download is ~110 MB), under a minute afterwards.

In [ ]:
# 2. setup: Drive, packages, data cache, W&B, env vars
from src.common.colab import setup
setup(fs2k=True)   # True for Task 4

## 3. Quick check (optional, ≈ 1 min)
Trains, searches, evaluates and exports on tiny synthetic paired data (W&B disabled, written to separate `smoke` folders, so real results are never touched). If this fails, fix the problem before spending GPU time.

In [ ]:
!python -m src.task4.train --smoke
!python -m src.task4.evaluate --smoke
!python -m src.task4.export_onnx --smoke

## 4. Optuna hyperparameter search (estimate ≈ 45–80 min)
Study `task4_cgan` tunes the seven hyperparameters the brief asks for: generator and discriminator learning rates, batch size, base channel count (shared by G and D), dropout rate, style-embedding dimension and the reconstruction weight λ_L1. Each trial trains **10** epochs instead of the final 120; weak trials are stopped by the median pruner; every trial is its own W&B run in group `task4-optuna`. Trial 0 is the brief's starting point (λ_L1 = 100, pix2pix learning rates) as a reference.

Trials are scored by validation `sketch_score = 0.5·SSIM + 0.5·(1 − L1)` on the **validation split only** — the official test set is never used for selection. The study stops after 20 finished trials or when no new trial may start after 80 minutes. **After a disconnect just run this cell again.** Results: `OUTPUT_DIR/task4/optuna/task4_cgan/` (plots, `trials.csv`, `search_space.json`) and the selected configuration `OUTPUT_DIR/task4/best_config.yaml`.

In [ ]:
!python -m src.task4.optuna_search --n-trials 20 --epochs 10 --timeout-min 80

## 5. Final training with the best configuration (estimate ≈ 25–60 min)
120 epochs with the Optuna configuration (constant learning rate for the first half, then linear decay to 0), W&B group `task4-final`. The log and W&B show **D real, D fake, G adversarial and G L1 separately**, per step and per epoch, plus validation L1/SSIM/PSNR, the sharpness diagnostic `edge_ratio` and the style-sensitivity gap. Every 5 epochs the **same fixed validation photos** are logged as a grid (photo | ground truth | the three styles), so the generator's development can be followed.

Checkpoints: `CKPT_DIR/task4/cgan/last.pt` (every epoch, G + D + optimisers) and `best.pt` (best validation `sketch_score`, uploaded to W&B as an artifact). Re-running this cell after a disconnect continues from the last finished epoch.

In [ ]:
!python -m src.task4.train --config $OUTPUT_DIR/task4/best_config.yaml --resume

## 6. Test-set evaluation (estimate ≈ 2 min)
All 1,046 official test pairs. Writes per-pair records, tables (CSV + LaTeX) **per style and per photo source**, the style-sensitivity numbers and the figures (typical results, style swap, failure cases) to `OUTPUT_DIR/task4/`. Per-source tables matter because style and source are confounded in the training data: styles 0/1 come only from photo1 and style 2 only from photo2/photo3, so parts of the test set are (source, style) combinations the model never saw (see `docs/fs2k_notes.md`).

In [ ]:
!python -m src.task4.evaluate

## 7. ONNX export for the application (estimate ≈ 1 min)
Exports **only the generator** (the discriminator is a training component) to `ONNX_DIR/generator.onnx`: opset 17, inputs `photo` [N,3,128,128] in [−1,1] and `style` int64 [N], output `sketch` [N,1,128,128] in [−1,1], dynamic batch. Checks that ONNX Runtime matches PyTorch on 64 real test photos (fails loudly otherwise) and writes the model card `generator.json`, including the preprocessing the backend must mirror. At `base_channels=64` the file is ~168 MB.

In [ ]:
!python -m src.task4.export_onnx

## Where the results are (on Google Drive, `MyDrive/GenAI_A1/`)
- `checkpoints/task4/cgan/best.pt` — final generator (and `last.pt` with the discriminator, for resuming)
- `optuna/task4_cgan.db` — the Optuna study; `outputs/task4/optuna/task4_cgan/` — its report
- `outputs/task4/best_config.yaml` — selected hyperparameters
- `outputs/task4/eval/`, `outputs/task4/tables/`, `outputs/task4/figures/` — test results for the report
- `onnx/generator.onnx` + `onnx/generator.json` — model for the application